<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1: Introducing Polars

## An ETL Showcase

### Extract

#### Import packages

In [ ]:
%%bash

pip install --upgrade polars

In [ ]:
import numpy as np

import polars as pl
from plotnine import *

In [ ]:
pl.__version__

#### Download and extract Citi Bike trips

In [ ]:
%%bash

curl -sO https://s3.amazonaws.com/tripdata/202403-citibike-tripdata.zip
unzip -o 202403-citibike-tripdata.zip -d sample_data/citibike/
rm -f 202403-citibike-tripdata.zip

#### Read Citi Bike trips into a Polars DataFrame

In [ ]:
! wc -l sample_data/citibike/202403-citibike-tripdata_?.csv

In [ ]:
! head -n 6 sample_data/citibike/202403-citibike-tripdata_1.csv

In [ ]:
trips = pl.read_csv(
    "sample_data/citibike/202403-citibike-tripdata_?.csv",
    schema_overrides={
        "started_at": pl.Datetime,
        "ended_at": pl.Datetime,
        "start_station_id": pl.String,
        "end_station_id": pl.String,
    },
).sort(
    "started_at"
)

trips.height

In [ ]:
print(trips)

#### Read in neighborhoods from GeoJSON

In [ ]:
!curl -so sample_data/citibike/nyc-neighborhoods.geojson https://raw.githubusercontent.com/davidrpugh/data-analytics-with-polars/refs/heads/main/data/citibike/nyc-neighborhoods.geojson

In [ ]:
! python -m json.tool sample_data/citibike/nyc-neighborhoods.geojson

In [ ]:
neighborhoods = (
    pl.read_json("sample_data/citibike/nyc-neighborhoods.geojson")
    .select("features")
    .explode("features")
    .unnest("features")
    .unnest("properties")
    .select("neighborhood", "borough", "geometry")
    .unnest("geometry")
    .with_columns(polygon=pl.col("coordinates").list.first())
    .select("neighborhood", "borough", "polygon")
    .filter(pl.col("borough") != "Staten Island")
    .sort("neighborhood")
)

neighborhoods

### Bonus: Visualizing Neighborhoods and Stations

In [ ]:
neighborhoods_coords = (
    neighborhoods.with_row_index("id")
    .explode("polygon")
    .with_columns(
        lon=pl.col("polygon").list.first(),
        lat=pl.col("polygon").list.last(),
    )
    .drop("polygon")
)

neighborhoods_coords

In [ ]:
stations = (
    trips.group_by(station=pl.col("start_station_name"))
    .agg(
        lon=pl.col("start_lng").median(),
        lat=pl.col("start_lat").median(),
    )
    .sort("station")
    .drop_nulls()
)
stations

In [ ]:
(
    ggplot(neighborhoods_coords, aes(x="lon", y="lat", group="id"))
    + geom_polygon(aes(alpha="neighborhood", fill="borough"), color="white")
    + geom_point(stations, size=0.1)
    + scale_x_continuous(expand=(0, 0))
    + scale_y_continuous(expand=(0, 0, 0, 0.01))
    + scale_alpha_ordinal(range=(0.3, 1))
    + scale_fill_brewer(type="qual", palette=2)
    + guides(alpha=False)
    + labs(
        title="New York City neighborhoods and Citi Bike stations",
        subtitle="2,143 stations across 106 neighborhoods",
        caption="Source: https://citibikenyc.com/system-data",
        fill="Borough",
    )
    + theme_void(base_size=14)
    + theme(
        dpi=300,
        figure_size=(7, 9),
        plot_background=element_rect(fill="white", color="white"),
        plot_caption=element_text(style="italic"),
        plot_margin=0.01,
        plot_title=element_text(ha="left"),
    )
)

### Transform

#### Clean up columns

In [ ]:
trips = trips.select(
    bike_type=pl.col("rideable_type")
    .str.split("_")
    .list.get(0)
    .cast(pl.Categorical),
    rider_type=pl.col("member_casual").cast(pl.Categorical),
    datetime_start=pl.col("started_at"),
    datetime_end=pl.col("ended_at"),
    station_start=pl.col("start_station_name"),
    station_end=pl.col("end_station_name"),
    lon_start=pl.col("start_lng"),
    lat_start=pl.col("start_lat"),
    lon_end=pl.col("end_lng"),
    lat_end=pl.col("end_lat"),
).with_columns(
    duration=(pl.col("datetime_end") - pl.col("datetime_start"))
)

trips.columns

#### Clean up rows

In [ ]:
trips = (
    trips.drop_nulls()
    .filter(
        (pl.col("datetime_start") >= pl.date(2024, 3, 1))
        & (pl.col("datetime_end") < pl.date(2024, 4, 1))
    )
    .filter(
        ~(
            (pl.col("station_start") == pl.col("station_end"))
            & (pl.col("duration").dt.total_seconds() < 5 * 60)
        )
    )
)

trips.height

#### Add trip distance

In [ ]:
# Haversine formula implemented in Polars expressions
def haversine_distance(lon1, lat1, lon2, lat2):
    # Earth radius in kilometers
    R = 6371.0

    # Convert degrees to radians
    lon1_rad = lon1 * np.pi / 180
    lat1_rad = lat1 * np.pi / 180
    lon2_rad = lon2 * np.pi / 180
    lat2_rad = lat2 * np.pi / 180

    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad

    a = (
        (dlat / 2).sin() ** 2
        + lat1_rad.cos() * lat2_rad.cos() * (dlon / 2).sin() ** 2
    )
    c = 2 * pl.arctan2(a.sqrt(), (1 - a).sqrt())
    return R * c

trips = trips.with_columns(
    distance=haversine_distance(
        pl.col("lon_start"),
        pl.col("lat_start"),
        pl.col("lon_end"),
        pl.col("lat_end")
    )
)

trips.select(
    "lon_start",
    "lon_end",
    "lat_start",
    "lat_end",
    "distance",
    "duration",
)

#### Add borough and neighborhood

In [ ]:
from shapely.geometry import Point, Polygon


def check_point_in_polygon(row):
    lon = row.get("lon")
    lat = row.get("lat")
    polygon_coords = row.get("polygon")
    if lon is None or lat is None or polygon_coords is None:
        return False
    point = Point(lon, lat)
    poly = Polygon(polygon_coords)
    return poly.contains(point)


stations = (
    stations.join(neighborhoods, how="cross")
    .with_columns(
        in_neighborhood=pl.struct(["lon", "lat", "polygon"])
        .map_elements(check_point_in_polygon, return_dtype=pl.Boolean)
    )
    .filter(pl.col("in_neighborhood"))
    .unique("station")
    .select(
        "station",
        "borough",
        "neighborhood",
    )
)

stations

In [ ]:
trips = (
    trips.join(
        stations.select(pl.all().name.suffix("_start")), on="station_start"
    )
    .join(stations.select(pl.all().name.suffix("_end")), on="station_end")
    .select(
        "bike_type",
        "rider_type",
        "datetime_start",
        "datetime_end",
        "duration",
        "station_start",
        "station_end",
        "neighborhood_start",
        "neighborhood_end",
        "borough_start",
        "borough_end",
        "lat_start",
        "lon_start",
        "lat_end",
        "lon_end",
        "distance",
    )
)

In [ ]:
print(trips[:, :4])
print(trips[:, 4:7])
print(trips[:, 7:11])
print(trips[:, 11:])

### Bonus: Visualizing Daily Trips per Borough

In [ ]:
trips_per_day = trips.group_by_dynamic(
    "datetime_start", group_by="borough_start", every="1d"
).agg(num_trips=pl.len())

trips_per_day

In [ ]:
from mizani.labels import label_comma
from mizani.breaks import breaks_date


(
    ggplot(
        trips_per_day,
        aes(x="datetime_start", y="num_trips", fill="borough_start"),
    )
    + geom_area()
    + scale_fill_brewer(type="qual", palette=2)
    + scale_x_datetime(breaks=breaks_date(width="1 day"), date_labels="%-d", expand=(0, 0))
    + scale_y_continuous(labels=label_comma(), expand=(0, 0))
    + labs(
        x="March 2024",
        fill="Borough",
        y="Trips per day",
        title="Citi Bike trips per day in March 2024",
        subtitle="On March 23, nearly 10 cm of rain fell in NYC",
    )
    + theme_tufte(base_size=14)
    + theme(
        axis_ticks_major=element_line(color="white"),
        figure_size=(8, 5),
        legend_position="top",
        plot_background=element_rect(fill="white", color="white"),
        plot_caption=element_text(style="italic"),
        plot_title=element_text(ha="left"),
    )
)

### Load

#### Write partitions

In [ ]:
trips_parts = (
    trips.sort("datetime_start")
    .with_columns(date=pl.col("datetime_start").dt.date().cast(pl.String))
    .partition_by(["date"], as_dict=True, include_key=False)
)

for key, df in trips_parts.items():
    df.write_parquet(f"sample_data/citibike/trips-{key[0]}.parquet")

#### Verify

In [ ]:
! ls -1 sample_data/citibike/*.parquet

In [ ]:
pl.read_parquet("sample_data/citibike/*.parquet").height

### Bonus: Becoming Faster by Being Lazy

In [ ]:
trips = (
    pl.scan_csv(
        "sample_data/citibike/202403-citibike-tripdata_?.csv",
        try_parse_dates=True,
        schema_overrides={
            "started_at": pl.Datetime,
            "ended_at": pl.Datetime,
            "start_station_id": pl.String,
            "end_station_id": pl.String,
        },
    )
    .select(
        bike_type=pl.col("rideable_type").str.split("_").list.get(0),
        rider_type=pl.col("member_casual"),
        datetime_start=pl.col("started_at"),
        datetime_end=pl.col("ended_at"),
        station_start=pl.col("start_station_name"),
        station_end=pl.col("end_station_name"),
        lon_start=pl.col("start_lng"),
        lat_start=pl.col("start_lat"),
        lon_end=pl.col("end_lng"),
        lat_end=pl.col("end_lat"),
    )
    .with_columns(duration=(pl.col("datetime_end") - pl.col("datetime_start")))
    .drop_nulls()
    .filter(
        ~(
            (pl.col("station_start") == pl.col("station_end"))
            & (pl.col("duration").dt.total_seconds() < 5 * 60)
        )
    )
    .with_columns(
        distance=haversine_distance(
            pl.col("lon_start"),
            pl.col("lat_start"),
            pl.col("lon_end"),
            pl.col("lat_end")
        )
    )
).collect()

neighborhoods = (
    pl.read_json("sample_data/citibike/nyc-neighborhoods.geojson")
    .lazy()
    .select("features")
    .explode("features")
    .unnest("features")
    .unnest("properties")
    .select("neighborhood", "borough", "geometry")
    .unnest("geometry")
    .with_columns(polygon=pl.col("coordinates").list.first())
    .select("neighborhood", "borough", "polygon")
    .sort("neighborhood")
    .filter(pl.col("borough") != "Staten Island")
)

stations = (
    trips.lazy()
    .group_by(station=pl.col("station_start"))
    .agg(
        lat=pl.col("lat_start").median(),
        lon=pl.col("lon_start").median(),
    )
    .with_columns(point=pl.concat_list("lon", "lat"))
    .drop_nulls()
    .join(neighborhoods, how="cross")
    .with_columns(
        in_neighborhood=pl.struct(["lon", "lat", "polygon"])
        .map_elements(check_point_in_polygon, return_dtype=pl.Boolean)
    )
    .filter(pl.col("in_neighborhood"))
    .unique("station")
    .select(
        pl.col("station"),
        pl.col("borough"),
        pl.col("neighborhood"),
    )
).collect()

trips = (
    trips.join(
        stations.select(pl.all().name.suffix("_start")), on="station_start"
    )
    .join(stations.select(pl.all().name.suffix("_end")), on="station_end")
    .select(
        "bike_type",
        "rider_type",
        "datetime_start",
        "datetime_end",
        "duration",
        "station_start",
        "station_end",
        "neighborhood_start",
        "neighborhood_end",
        "borough_start",
        "borough_end",
        "lat_start",
        "lon_start",
        "lat_end",
        "lon_end",
        "distance",
    )
)

trips.height